# 10b · 对外 API：FastAPI → 容器 → Lambda → API Gateway

## 这一章在整个项目里的位置

```
00–09 ✅ → [10 对外服务] → 11 RAG → 12 Agent → 13 收尾
              │
              ├─ 10a 数据湖 + Athena ✅
              └─ 10b API：问一个突变，一两秒内回答   ← 你在这里
```

## 背景：为什么需要这一章

10a 之后，会写 SQL 的人能查分数了。但大多数"用户"不写 SQL：
- 一个网页想显示 TP53 R175H 的分数
- 一个临床流程想批量查病人的突变
- 12 号的 Agent 想在回答问题时调用我们的模型

它们都需要一个 **API**：用一个网址问，拿回一段标准格式（JSON）的回答。比如：

```
GET /v1/proteins/TP53/variants/R175H
→ {"variant": "R175H", "stability_score": 0.22, "function_score": 0.30, ...}
```

这一章把 API 写出来、打包、部署到 AWS。**这是 ML 工程师"把模型交到用户手里"的最后一步**。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 2 步：在你的 Mac 上跑 API（用你的 AWS 身份查 Athena）
   ├─ 第 3 步：打包成容器镜像（Colima，约 1–3 分钟）
   ├─ 第 4、5 步：推送镜像，Terraform 部署 Lambda 和 API Gateway
   └─ 第 6、7 步：调用线上的 API，看日志
看结果：AWS 控制台 → Lambda → qafi-dev-api；API Gateway → qafi-dev-api
```

## 这一章做的 7 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 设计 API | 定好菜单：能点什么、怎么点、点错了怎么说 |
| 2 | 在本机跑 | 在自家厨房试做 |
| 3 | 打包成容器 | 装进标准的餐盒，到哪儿都一样 |
| 4 | 推送到 ECR | 餐盒放进中央仓库 |
| 5 | 部署 Lambda + API Gateway | 开一个有门卫的取餐窗口 |
| 6 | 调用线上 API | 真的去窗口点一份 |
| 7 | 看日志 | 翻看出餐记录 |

**花费**：几乎为 0（Lambda、API Gateway 每月前 100 万次请求在免费额度内；镜像存储每月约 0.02 美元）。**时间**：约 40 分钟。

**开始前**：10a 已经跑完；在终端运行 `aws login`。

---

## 准备

运行下面这一格。**不用看代码。**

In [ ]:
import json, os, subprocess, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import boto3
import pandas as pd
from fastapi.testclient import TestClient
from qafi_platform import api, api_client
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(AWS_PROFILE="qafi", AWS_DEFAULT_REGION="eu-central-1")
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
def terraform_output(name):
    return json.loads(subprocess.run(f"terraform -chdir=infra/dev output -json {name}", shell=True, capture_output=True, text=True).stdout)
aws = boto3.Session(region_name="eu-central-1")
LAKE = terraform_output("lake")
os.environ.update(ATHENA_WORKGROUP=LAKE["workgroup"], ATHENA_DATABASE=LAKE["database"])
print("AWS:", aws.client("sts").get_caller_identity()["Arn"], "| 数据湖:", LAKE)

---

## 第 1 步 · 设计 API

### 在干什么
定下 API 有哪些地址、怎么问、出错时怎么回答。

### 背景
比喻：**定好菜单**。菜单要让人一看就会点，点错了要说清楚错在哪。

我们的 API 只有四个地址（代码在 `src/qafi_platform/api.py`）：

| 地址 | 回答什么 | 例子 |
|---|---|---|
| `GET /health` | 服务活着吗（不查数据） | `{"status": "ok"}` |
| `GET /v1/proteins/{protein}` | 一个蛋白：基因名、长度、哪一版数据 | `/v1/proteins/TP53` |
| `GET /v1/proteins/{protein}/variants/{variant}` | 一个突变的两个分数 | `/v1/proteins/TP53/variants/R175H` |
| `GET /v1/proteins/{protein}/positions/{position}` | 一个位置的全部 19 种替换 | `/v1/proteins/TP53/positions/175` |

几个设计上的讲究：
- **按"资源"组织地址**（REST 风格）：蛋白 → 它的突变。地址读起来就是意思
- **`/v1/`**：版本号。以后要改回答的格式，就开 `/v2/`，用 `/v1/` 的程序不受影响
- `{protein}` 可以是基因名（TP53，大小写都行），也可以是 UniProt 编号（P04637）
- **出错要说清楚**，用标准的 HTTP 状态码：

| 情况 | 状态码 | 回答 |
|---|---|---|
| 正常 | 200 | 分数 |
| 没有这个蛋白 | 404 | `no protein named TP99` |
| 位置超出长度 | 404 | `TP53 has 393 residues` |
| 原来的氨基酸写错了 | 422 | `position 175 of TP53 is R, not G` |
| 格式不对（比如 R175X） | 422 | 哪个参数不符合规则 |
| 一个基因名对应几个蛋白 | 409 | 列出编号，请用编号问 |

**输入先检查，再查数据**：格式不对的请求根本到不了 Athena。查询时，值也是作为"参数"传给 Athena，不拼进 SQL 文本里。这样即使有人在网址里塞进一段 SQL，也只会被当成一个查不到的名字。这叫防 **SQL 注入**。

### 用了什么技术，为什么
- **FastAPI**：Python 里最常用的 API 框架之一。写一个函数，标注好参数的类型和规则，它自动检查输入、生成交互式文档
- **Pydantic**：定义回答的格式（哪些字段、什么类型）。文档里就能看到每个字段的说明
- **OpenAPI**：API 说明书的行业标准格式，FastAPI 自动生成。别的程序（包括 12 号的 Agent）读它就知道怎么调用

### 执行
看看 FastAPI 自动生成的说明书里有哪些地址：

In [ ]:
pd.DataFrame([{"地址": path, "参数": ", ".join(p["name"] for p in op["get"].get("parameters", []))}
              for path, op in api.app.openapi()["paths"].items()])

### 公司里什么情况怎么做
- API 先设计、后实现：地址、参数、回答格式、错误码定好，写进 OpenAPI，前后端各自开工
- 永远不要把用户输入拼进 SQL；永远先验证输入

### 面试怎么说
> "A small REST API, versioned under /v1, resource-oriented URLs, validated inputs and explicit error codes; FastAPI generates the OpenAPI contract. Values go to Athena as execution parameters, never concatenated into SQL."

---

## 第 2 步 · 在本机跑

### 在干什么
在你的 Mac 上运行 API，用你的 AWS 身份去查 10a 的数据湖。

### 背景
比喻：**在自家厨房试做**。上线前先在本机确认：回答对不对、错误信息清不清楚、多快。

**一次回答的过程**（以 TP53 R175H 为例）：
1. 检查格式：TP53 是合法的名字，R175H 是合法的突变写法
2. 查 `proteins_current`：TP53 → P04637，长度 393
3. 查 `variant_effect_current`：P04637 第 175 位的 19 行（10a 说过：按编号查，只读一小段）
4. 核对第 175 位确实是 R，挑出 H 那一行，回答

**两层缓存**：同样的问题第二次问，
- 函数还开着时，答案直接从**内存**里拿（毫秒）
- 内存里没有，**Athena 会复用 60 分钟内同样查询的结果**，不再扫描，也不收费

### 执行

In [ ]:
local = TestClient(api.app)
for path in ["/health", "/v1/proteins/tp53", "/v1/proteins/TP53/variants/R175H", "/v1/proteins/TP53/variants/R175H",
             "/v1/proteins/TP53/variants/G175H", "/v1/proteins/TP53/variants/R175X", "/v1/proteins/TP99/variants/R175H"]:
    started = time.time()
    response = local.get(path)
    detail = response.json().get("detail", "")
    print(f"{response.status_code}  {time.time() - started:5.2f} s  {path}  {str(detail)[:70]}")
local.get("/v1/proteins/TP53/variants/R175H").json()

### 结果怎么看
每一行：状态码、用时、地址、出错时的说明。每行前面那行 `{"path": …, "status": …, "ms": …}` 是 API 自己写的日志，第 7 步会在 CloudWatch 里看到一模一样的行。

写这一章时的结果：
- `/health`：几毫秒，不查数据
- 第一次问 TP53：约 1.5 秒（查了一次 `proteins_current`）
- 第一次问 R175H：约 1.3 秒（蛋白已经知道了，只查这个位置）
- **同样的问题再问一次：0.00 秒**，答案在内存里
- G175H：422，并说清楚第 175 位是 R
- R175X：422，格式不对（X 不是氨基酸），根本没去查数据
- TP99：404，没有这个基因

最后是 R175H 的完整回答：两个分数、ESM 零样本分，以及**这个分数来自哪一版数据、哪一版模型**。每个回答都可以追溯来源。

### 面试怎么说
> "Each answer carries the data release and the registry versions of both models, so any score a user sees can be traced back to the exact model and training data."

---

## 第 3 步 · 打包成容器

### 在干什么
把 API 和它需要的一切装进一个容器镜像，在你的 Mac 上启动它试一下。

### 背景
比喻：**装进标准的餐盒**。在你的 Mac 上能跑，换到 AWS 就不一定：Python 版本、装的包都可能不同。**容器**把代码和它需要的全部环境一起打包，在哪儿跑都一样。02 号已经做过一次。

这个镜像（`containers/api.Dockerfile`）有四个讲究：
1. **只装 API 需要的包**（FastAPI、uvicorn、boto3），不装 pandas、PyTorch。镜像小，启动快
2. **两段式构建**：第一段用带 uv 的镜像把依赖装好；最终的镜像从干净的官方 Python 镜像重新开始，只拷进装好的环境和我们的代码。构建工具不进生产环境
3. **为 Graviton（arm64）处理器构建**：AWS 自己的芯片，比同档的 x86 便宜约 20%。你的 Mac 也是 arm64，正好
4. **Lambda Web Adapter**：AWS 官方的一个小插件。Lambda 本来不是 Web 服务器，这个插件把每次调用翻译成普通的网页请求，交给 uvicorn。所以**同一个镜像**，在你的 Mac 上就是一个普通的 Web 服务，放进 Lambda 也能用，代码里没有一行是专门为 Lambda 写的

**Colima**：在 Mac 上运行容器的工具（一个轻量的 Docker），用完就关，省内存。

### 执行
启动 Colima，构建镜像（第一次约 1–3 分钟，要下载基础镜像）：

In [ ]:
run("colima start --cpu 2 --memory 2 2>&1 | tail -1")
started = time.time()
# Lambda wants a plain single-platform image: BuildKit (buildx) would add a provenance record unless told not to
provenance = "--provenance=false " if subprocess.run("docker buildx version", shell=True, capture_output=True).returncode == 0 else ""
run(f"docker build -q {provenance}--platform linux/arm64 -f containers/api.Dockerfile -t qafi-api .")
size = subprocess.run("docker image inspect qafi-api --format '{{.Size}}'", shell=True, capture_output=True, text=True).stdout
print(f"构建用了 {time.time() - started:.0f} 秒，镜像 {int(size) / 1e6:.0f} MB")

在你的 Mac 上启动这个容器，让它用你的 AWS 身份查一次真实数据（身份通过环境变量交给容器，不会打印出来）：

In [ ]:
credentials = subprocess.run("aws configure export-credentials --format env-no-export", shell=True,
                             capture_output=True, text=True).stdout.split()
container_env = {**os.environ, **dict(line.split("=", 1) for line in credentials)}
subprocess.run("docker rm -f qafi-api-test", shell=True, capture_output=True)
subprocess.run("docker run -d --name qafi-api-test -p 8080:8080 -e AWS_ACCESS_KEY_ID -e AWS_SECRET_ACCESS_KEY "
               "-e AWS_SESSION_TOKEN -e AWS_DEFAULT_REGION -e ATHENA_WORKGROUP -e ATHENA_DATABASE qafi-api",
               shell=True, env=container_env, check=True, capture_output=True)
time.sleep(3)
for path in ["/health", "/v1/proteins/TP53/variants/R175H"]:
    print(path, "→", subprocess.run(f"curl -s localhost:8080{path}", shell=True, capture_output=True, text=True).stdout[:200])
run("docker rm -f qafi-api-test")

### 结果怎么看
- 镜像约 68 MB
- 容器里的 API 和第 2 步回答一样：同一份代码，换了一个环境照样工作

### 公司里什么情况怎么做
- 每个服务一个尽量小的镜像，只装它需要的东西：小，就启动快、漏洞少
- 镜像在 CI 里也会构建一次（`.github/workflows/ci.yml` 的 docker 任务：构建、启动、检查 `/health`）

### 面试怎么说
> "The API is a FastAPI app in a slim arm64 image with only its own dependencies. The AWS Lambda Web Adapter lets the same image run as a plain web server locally and inside Lambda, with no Lambda-specific code."

---

## 第 4 步 · 推送到 ECR

### 在干什么
把镜像推送到 AWS 的镜像仓库 ECR，记下它的唯一编号。

### 背景
比喻：**餐盒放进中央仓库**。Lambda 只能从 ECR（Elastic Container Registry）里取镜像。10a 的 Terraform 已经建好了仓库 `qafi-dev-api`，规则是：
- **标签不可改**（immutable）：一个标签永远对应同一个镜像。我们用"代码版本 + 时间"做标签
- **推送时自动扫描漏洞**：检查镜像里的系统包和 Python 包有没有已知的安全问题
- **只留最新 5 个**：够回滚，又不会越存越多

推送后，记下镜像的**摘要**（digest，`sha256:…`）。标签只是一个名字，摘要才是镜像内容本身的指纹：内容差一个字节，摘要就完全不同。部署时用摘要，就保证上线的是**这一个**镜像。

### 执行

In [ ]:
API = terraform_output("api")
REPOSITORY = API["repository"]
COMMIT = subprocess.run("git rev-parse --short HEAD", shell=True, capture_output=True, text=True).stdout.strip()
TAG = f"{COMMIT}-{time.strftime('%Y%m%d-%H%M')}"
run(f"aws ecr get-login-password | docker login --username AWS --password-stdin {REPOSITORY.split('/')[0]} 2>&1 | tail -1")
run(f"docker tag qafi-api {REPOSITORY}:{TAG} && docker push -q {REPOSITORY}:{TAG}")
ecr = aws.client("ecr")
digest = ecr.describe_images(repositoryName="qafi-dev-api", imageIds=[{"imageTag": TAG}])["imageDetails"][0]["imageDigest"]
IMAGE = f"{REPOSITORY}@{digest}"
Path("infra/dev/build/api_image.txt").write_text(IMAGE + "\n")
print("要部署的镜像：", IMAGE)
for attempt in range(30):  # the scan starts a few seconds after the push and takes a few more
    try:
        scan = ecr.describe_image_scan_findings(repositoryName="qafi-dev-api", imageId={"imageDigest": digest})
        if scan["imageScanStatus"]["status"] == "COMPLETE":
            break
    except ecr.exceptions.ScanNotFoundException:
        pass
    time.sleep(10)
print("漏洞扫描：", scan["imageScanStatus"]["status"], scan["imageScanFindings"].get("findingSeverityCounts", {}))

### 结果怎么看
- `要部署的镜像`：`…/qafi-dev-api@sha256:…`。这一行也写进了 `infra/dev/build/api_image.txt`，下一步 Terraform 读它
- `漏洞扫描`：按严重程度数已知漏洞。写这一章时：0 个严重（CRITICAL）、3 个高危（HIGH，在系统的 glibc 和 zlib 里），没有一个出在我们装的 Python 包里

**一个真实的教训**：这个镜像一开始用的是 Debian 12 的基础镜像，扫描出 **10 个严重、47 个高危**漏洞，全部在 perl、openssl、gnutls 这些老的系统包里。换成当前的 Debian 13，再用两段式构建去掉构建工具，降到 0 个严重、3 个高危。镜像里的漏洞，大部分来自你**没有写的那部分**：基础镜像。所以公司里会定期重新构建镜像，好拿到打过补丁的基础镜像

---

## 第 5 步 · 部署 Lambda + API Gateway

### 在干什么
用 Terraform 建 Lambda 函数和 API Gateway，把镜像接到网上。

### 背景
比喻：**开一个有门卫的取餐窗口**。三样东西：

**1. Lambda 函数 `qafi-dev-api`**：运行我们的镜像。
- 有请求才启动，没请求不花钱
- 第一次请求（或闲置一段时间后）要先启动容器，叫**冷启动**，多花一两秒；之后几分钟内的请求直接用热着的容器
- 它的身份（IAM 角色）只能做三件事：在 `qafi-dev` 工作组里查询、读数据湖里那两张表的文件、写查询结果。**多一点都不能**：不能删文件，不能读 raw 桶，不能碰别的服务

**2. API Gateway**：门卫，挡在 Lambda 前面。
- **只放行签了名的请求**（AWS_IAM 授权）：请求要用有权限的 AWS 身份签名，你的账号可以，网上的陌生人不行。这样没人能用你的账单跑查询
- **限流**：平均每秒最多 2 个请求，同时最多 5 个。程序出错、无限循环地调用，也花不了多少钱
- **访问日志**：每个请求一行：谁、问了什么、回答状态、多久

**3. 连线**：API Gateway 收到 `GET /…` 就交给 Lambda，Lambda 的回答原样返回。

### 执行

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev plan -input=false -no-color -out=dev.tfplan 2>&1 | grep -E 'will be|Plan:|No changes|Error'")

`Plan: 6 to add, 0 to change, 0 to destroy`：函数、API、连线、路由、阶段（stage）、调用权限。确认后执行：

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan 2>&1 | grep -E 'complete|Error'")
API = terraform_output("api")
for attempt in range(30):  # a new API answers "Not Found" for a few seconds, until its first deployment is live
    if api_client.get(aws, f"{API['url']}/health")[0] == 200:
        break
    time.sleep(2)
API

### 结果怎么看
`url` 就是 API 的网址：`https://<编号>.execute-api.eu-central-1.amazonaws.com`。新建的 API 要几秒钟才生效（之前会回答 "Not Found"），所以这一格最后会等它能回答 `/health` 了再结束。

### 去网页上看
- AWS 控制台 → **Lambda** → `qafi-dev-api`：Image 一栏是我们的镜像摘要；Configuration → Permissions 能看到它的角色和那几条权限
- **API Gateway** → `qafi-dev-api` → Routes：`GET /{proxy+}`，Authorization 是 IAM

### 公司里什么情况怎么做
- **每个函数一个专属角色，只给它需要的权限**（最小权限），这是 AWS 安全审查最先看的地方
- 对外的 API 一定有门卫：身份验证（IAM、API 密钥或登录令牌）+ 限流
- 基础设施（函数、API、权限）用 Terraform 管；**代码的新版本**只是换一个镜像摘要再 apply 一次

### 面试怎么说
> "Lambda runs the image on demand behind an HTTP API in API Gateway, with IAM authorization and throttling at 2 requests per second. The function's role can only query one Athena workgroup and read two prefixes of the lake: least privilege."

---

## 第 6 步 · 调用线上 API

### 在干什么
用你的 AWS 身份签名，真的去调用线上的 API。

### 背景
**签名**是怎么回事：请求发出前，用你的 AWS 密钥对"这个请求的内容 + 时间"算一个签名，附在请求头里。API Gateway 用同样的方法验算，对上了才放行。
- 密钥本身**从不发送**
- 签名只对**这一个请求**、在**几分钟内**有效，被人截获也没用

同一个问题问三次，看冷启动；最后再发一个**不签名**的请求，看门卫拦不拦。

### 执行

In [ ]:
URL = API["url"]
for attempt in ["第一次（冷启动）", "第二次", "第三次"]:
    status, body, seconds = api_client.get(aws, f"{URL}/v1/proteins/TP53/variants/R175H")
    print(f"{attempt}: {status}  {seconds:.2f} 秒  stability={body.get('stability_score')}  function={body.get('function_score')}")
status, body, seconds = api_client.get(aws, f"{URL}/v1/proteins/BRCA1/positions/1699")
print(f"BRCA1 第 1699 位：{status}，{len(body.get('variants', []))} 种替换，{seconds:.2f} 秒")
status, body, _ = api_client.get(aws, f"{URL}/v1/proteins/TP53/variants/R175H", sign=False)
print(f"不签名：{status} {body}")

### 结果怎么看
写这一章时的结果：
- **第一次约 3 秒**：函数刚启动，内存里还没有答案，要连查两次 Athena。真正的"冷启动"（启动容器、Python、FastAPI）约 2.2 秒，发生在上一格等 `/health` 的时候，第 7 步的日志里叫 `Init Duration`
- **第二、三次约 0.35 秒**：答案在函数的内存里，时间几乎都花在网络和签名验证上
- **BRCA1 第 1699 位**：200，19 种替换
- **不签名：403 Forbidden**。门卫在 Lambda 之前就挡下了，函数根本没被调用（第 7 步访问日志里这一行的 `caller` 和 `lambdaMs` 都是 "-"）

---

## 第 7 步 · 看日志

### 在干什么
看 Lambda 和 API Gateway 记下了什么。

### 背景
比喻：**翻看出餐记录**。线上服务出了问题，第一件事就是看日志。我们有两份：
- **Lambda 的日志**：我们的程序每个请求写一行 JSON（地址、状态、毫秒），还有 Lambda 自己的启动记录（冷启动用了多久）
- **API Gateway 的访问日志**：每个请求一行，包括**谁**调用的（AWS 身份）

日志都在 CloudWatch Logs，保存 30 天后自动删除。13 号会在日志上加告警。

### 执行

In [ ]:
run("aws logs tail /aws/lambda/qafi-dev-api --since 15m --format short | grep -E 'path|Init|REPORT' | tail -8")
run("aws logs tail /aws/apigateway/qafi-dev-api --since 15m --format short | tail -4")

### 结果怎么看
- `{"path": "/v1/proteins/TP53/variants/R175H", "status": 200, "ms": …}`：我们写的那一行
- `REPORT … Init Duration: … ms`：只在冷启动的那次出现，是启动容器用的时间（写这一章时约 2.2 秒）。`Max Memory Used` 约 110 MB：分配的 512 MB 绰绰有余，留着是因为 Lambda 按内存比例分配处理器，内存多，启动也快
- 访问日志里的 `caller`：你的 AWS 用户。每个请求都知道是谁发的

### 发新版本和回滚
以后改了 API 的代码：重新运行第 3、4、5 步。新镜像有新的摘要，Terraform 只改一处：函数用的镜像。
- **回滚**：把 `api_image.txt` 换回上一个镜像的摘要（ECR 里留着最新 5 个），再 apply
- **数据回滚**（10a）：把 `*_current` 视图指回上一版。API 不用动

最后关掉 Colima，把内存还给 Mac：

In [ ]:
run("colima stop 2>&1 | tail -1")

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 10b prediction API: FastAPI on Lambda behind API Gateway") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿：docker 任务现在也会构建 API 镜像并检查 `/health`。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 设计 | 4 个地址，`/v1/`，清楚的错误码 | 先验证输入；值作为参数，不拼进 SQL |
| 2 本机 | TestClient 查真实数据 | 两层缓存：内存 + Athena 结果复用 |
| 3 容器 | 只装 API 的包，arm64，Lambda Web Adapter | 同一个镜像，本机和 Lambda 都能跑 |
| 4 ECR | 标签不可改，推送即扫描，用摘要部署 | 摘要是镜像内容的指纹 |
| 5 部署 | Lambda + API Gateway，IAM 授权 + 限流 | 最小权限；门卫 |
| 6 调用 | 签名请求；冷启动约 2–3 秒，之后约 0.35 秒 | 不签名就被拦下 |
| 7 日志 | 程序日志 + 访问日志 | 出问题先看日志 |

**一句话**：从原始数据到对外服务的整条链路通了：任何有权限的程序，用一个网址就能问到人类蛋白组里任何一个错义突变的分数，并且知道它来自哪一版数据和模型。

## 面试题

1. **Walk me through your serving architecture.**
   > "Scores are batch-computed in Databricks, exported to S3 as sorted Parquet and registered in Glue. A FastAPI app in a container runs on Lambda behind API Gateway with IAM auth and throttling; it queries Athena with parameterised SQL and caches answers."

2. **Why Lambda and not a container service like ECS or a server?**
   > "Traffic is low and bursty: Lambda costs nothing when idle. The cost is a cold start of about a second; for steady high traffic, the same image would run on ECS Fargate unchanged."

3. **How do you deploy and roll back?**
   > "Immutable image tags in ECR, deployment by digest through Terraform; rollback is the previous digest. Data releases roll back independently by moving the Athena views."

4. **How is the API secured?**
   > "IAM-signed requests only, throttling at the gateway, input validation before any query, parameterised SQL, and a function role that can read two prefixes of one bucket and query one workgroup."

**下一步 11 · RAG**：用 Bedrock 的知识库收录 UniProt 的蛋白注释和你的论文，让 Claude 能回答"这个蛋白是做什么的、这个突变为什么可能有害"，并给出处。